# Multi-process Orchestrator for Remaining Work

This notebook runs only the specific folds that were stuck or failed, using the flattened task architecture to avoid bottlenecks.

In [1]:
import os
import subprocess
import concurrent.futures
from tqdm.notebook import tqdm
import sys

# Configure paths
EXPERIMENT_SCRIPT = os.path.join('..', 'experiment.py')
LOG_DIR = 'logs'
os.makedirs(LOG_DIR, exist_ok=True)

MAX_WORKERS = 7
N_SPLITS = 5

REMAINING_TARGETS = [
    ('Plant_oil', 20),
    ('Plant_oil', 10),
    ('Plant_oil', 5),
    ('Chinese_wine', 20),
    ('Chinese_wine', 10)
]

def run_task(task_kwargs):
    dataset = task_kwargs['dataset']
    seed = task_kwargs['seed']
    fold = task_kwargs['fold']
    pbar = task_kwargs['pbar']
    log_file = os.path.join(LOG_DIR, f"{dataset}_S{seed}_F{fold}.log")
    
    with open(log_file, "w", encoding="utf-8") as f:
        f.write(f"Starting {dataset} - Seed {seed} - Fold {fold}\n\n")
        f.flush()
        
        command = [sys.executable, EXPERIMENT_SCRIPT, "--dataset", dataset, "--seed", str(seed), "--fold", str(fold), "--n_selections", "5"]
        process = subprocess.Popen(command, stdout=f, stderr=subprocess.STDOUT, text=True)
        process.wait()
        
        pbar.update(1)
        
        if process.returncode != 0:
            return f"[!] ERROR in {dataset} (Seed {seed}, Fold {fold}) - See details at {log_file}"
                
    return f"[+] COMPLETED {dataset} (Seed {seed}, Fold {fold})"

In [2]:
# Create task list and shared progress bar
total_folds = len(REMAINING_TARGETS) * N_SPLITS
pbar = tqdm(total=total_folds, desc="Total Remaining Folds Progress")

tasks = []
for dataset, seed in REMAINING_TARGETS:
    for fold in range(1, N_SPLITS + 1):
        tasks.append({'dataset': dataset, 'seed': seed, 'fold': fold, 'pbar': pbar})

print(f"Total independent folds to run: {len(tasks)}")
print(f"Parallel workers: {MAX_WORKERS}")
print("Starting... Output will be written to Runners/logs/")

results = []
with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(run_task, task) for task in tasks]
    for future in concurrent.futures.as_completed(futures):
        try:
            res = future.result()
            print(res)
            results.append(res)
        except Exception as e:
            print(f"[!!!] Process exception: {e}")

pbar.close()
print("\n=== ALL REMAINING TASKS COMPLETED ===")

Total Remaining Folds Progress:   0%|          | 0/25 [00:00<?, ?it/s]

Total independent folds to run: 25
Parallel workers: 7
Starting... Output will be written to Runners/logs/
[+] COMPLETED Plant_oil (Seed 10, Fold 1)
[+] COMPLETED Plant_oil (Seed 10, Fold 2)
[+] COMPLETED Plant_oil (Seed 20, Fold 4)
[+] COMPLETED Plant_oil (Seed 20, Fold 5)
[+] COMPLETED Plant_oil (Seed 20, Fold 3)
[+] COMPLETED Plant_oil (Seed 20, Fold 2)
[+] COMPLETED Plant_oil (Seed 10, Fold 3)
[+] COMPLETED Plant_oil (Seed 20, Fold 1)
[+] COMPLETED Plant_oil (Seed 10, Fold 4)
[+] COMPLETED Plant_oil (Seed 5, Fold 2)
[+] COMPLETED Plant_oil (Seed 5, Fold 1)
[+] COMPLETED Plant_oil (Seed 5, Fold 3)
[+] COMPLETED Plant_oil (Seed 5, Fold 5)
[+] COMPLETED Plant_oil (Seed 5, Fold 4)
[+] COMPLETED Plant_oil (Seed 10, Fold 5)
[+] COMPLETED Chinese_wine (Seed 10, Fold 1)
[+] COMPLETED Chinese_wine (Seed 20, Fold 1)
[+] COMPLETED Chinese_wine (Seed 10, Fold 2)
[+] COMPLETED Chinese_wine (Seed 20, Fold 3)
[+] COMPLETED Chinese_wine (Seed 20, Fold 2)
[+] COMPLETED Chinese_wine (Seed 20, Fold 4